# 01 · What is in BigEarthNet.txt, and what we may safely train on

This notebook is the *look before you train* step. Fine-tuning a model on data you have not examined is
how you get a model that scores 95% on the validation set and answers "yes" to everything in the demo.
Three questions, in order:

1. **What is the data?** Rows, types, categories, splits.
2. **What could a lazy model exploit?** Answer priors (is "yes" 80% of the answers?), constant fields,
   template repetition.
3. **What do we actually have images for?** BigEarthNet.txt is text only; the pictures are BigEarthNet
   v2.0's, 145 GB in full. We use the published Lithuania-summer subset (8,775 patches, 2.3 GB) - and that
   choice has consequences the model must not be taught.

Everything decided here is enforced in `training/vlm/prepare_bigearthnet_txt.py` and named in
`app/constants/vlm.py`; the notebook explains, the code enforces.

In [ ]:
import sys, pathlib
BACKEND = pathlib.Path.cwd().resolve()
while BACKEND.name != "backend":
    BACKEND = BACKEND.parent
sys.path.insert(0, str(BACKEND))
import pandas as pd, numpy as np
pd.set_option("display.width", 160, "display.max_colwidth", 120)

text = pd.read_parquet(BACKEND / "data/datasets/bigearthnet-txt/BigEarthNet.txt.parquet")
print(f"{len(text):,} rows, {text.patch_id.nunique():,} S2 patches, columns: {list(text.columns)}")

## 1. Shape of the data

Four answer *types* and eleven *categories*. `split` has the usual train/validation/test plus a fourth,
`bench`: 15,029 rows over 1,082 pairs that humans verified. Everything else was generated by templates
from the land-cover map and then paraphrased by an LLM - which is fine to *train* on and wrong to
*report* on. Numbers we publish come from `bench` only.

In [ ]:
print(text["split"].value_counts().to_frame("rows"))
print(text["type"].value_counts().to_frame("rows"))
print(text.groupby(["category", "type"]).size().unstack(fill_value=0))

## 2. What a lazy model could exploit

**Answer priors.** For yes/no questions, if 80% of answers were "yes", a model that never looks at the
image scores 80%. Here binary is 51/49 and MCQ is a flat 25% per letter: a good sign, and the reason
exact-match accuracy is a fair metric on this data. (RSVQA-LR, by contrast, is famously skewed.)

**Constant fields in our image subset.** The images we can afford are *one country, one season*. Three
categories - `country`, `season`, `climate zone` - therefore have a single answer in every row we can show
the model. Train on them and the model learns "Lithuania" and "summer" as reflexes, and will say so about
a Cartosat scene of Gujarat. They are dropped from training (`BEN_TXT_CONSTANT_CATEGORIES`).

**Template repetition.** 375,926 distinct captions over 463,932 rows, and the most common caption occurs
9,700 times ("...entirely dominated by marine waters..."). Captions are capped per bucket like everything
else, and the eval reports ROUGE-L, which a memorised template cannot game on unseen patches.

In [ ]:
for kind in ["binary", "mcq"]:
    share = text.loc[text.type == kind, "output"].value_counts(normalize=True).round(3)
    print(kind, share.head(4).to_dict())
captions = text.loc[text.type == "captioning", "output"]
print(f"captions: {captions.nunique():,} distinct of {len(captions):,}; most common repeats {captions.value_counts().iloc[0]:,} times")
print(text[text.type == "mcq"].sample(3, random_state=1)[["category", "input", "output"]].to_string())

## 3. What we have images for

`patch_id` keys into BigEarthNet v2.0. The Lithuania-summer LMDB (Hugging Face, `hackelle/...`) holds
8,775 patches with both S1 and S2; 8,363 of them have BigEarthNet.txt rows. Those rows are ~185k, of which
**207 are `bench`** - small, but they are the only human-verified rows we can score with pictures, and
they are reported beside the larger template-generated `test` sample with that caveat stated.

Patches flagged as cloudy or snow-covered are excluded: the caption says "arable land" and the picture
shows a cloud, and a model trained on that pair learns to hallucinate.

In [ ]:
import glob
root = glob.glob(str(BACKEND / "data/models/datasets--hackelle--BigEarthNetV2-Lithuania-Summer-LMDB/snapshots/*"))[0]
meta = pd.read_parquet(f"{root}/metadata_lithuania_summer.parquet")
usable = meta[~meta.contains_cloud_or_shadow & ~meta.contains_seasonal_snow]
ours = text[text.patch_id.isin(set(usable.patch_id))]
print(f"Lithuania patches {len(meta):,}, usable {len(usable):,}; BigEarthNet.txt rows on them {len(ours):,}")
print(ours.split.value_counts().to_frame("rows"))
from app.constants.vlm import BEN_TXT_CONSTANT_CATEGORIES, BEN_TXT_TRAINED_CATEGORIES
print("trained categories:", sorted(BEN_TXT_TRAINED_CATEGORIES))
print("dropped as constant in this subset:", {c: ours.loc[ours.category == c, "output"].nunique() for c in BEN_TXT_CONSTANT_CATEGORIES})

## 4. What the model will see

A BigEarthNet patch is 120×120 pixels at 10 m - 1.2 km on a side. The VLM's vision encoder works in
16-pixel patches; at 120 px that is 7×7 = 49 patches, too coarse to see a road. We upscale to 448 px on
the longer side (28×28 patches, 196 visual tokens after the 2×2 merge). Upscaling adds no information,
but it lets the encoder spend more tokens on the information that is there.

S2 is rendered as true colour with a fixed 0-3000 reflectance window; S1 as a false-colour composite
(VV, VH, VV/VH in dB). Fixed windows, never per-image percentiles: the same field must render the same on
two dates, and a percentile stretch on a uniform field amplifies noise into texture.

In [ ]:
import lmdb
from safetensors.numpy import load as load_safetensors
from PIL import Image
from app.services.vlm.math.rendering import render_s1_false_colour, render_s2_true_colour
from app.models.vlm import fit_image

env = lmdb.open(f"{root}/BENv2_lithuania_summer.lmdb", readonly=True, lock=False)
row = ours[ours.type == "captioning"].iloc[0]
with env.begin() as txn:
    s2 = load_safetensors(txn.get(row.patch_id.encode()))
    s1 = load_safetensors(txn.get(row.s1_name.encode()))
rgb = render_s2_true_colour(s2["B04"], s2["B03"], s2["B02"])
sar = render_s1_false_colour(s1["VV"], s1["VH"])
print("S2 patch", rgb.shape, "-> shown to the model at", fit_image(rgb).size)
print("caption:", row.output[:300], "...")
Image.fromarray(np.concatenate([rgb, sar], axis=1)).resize((720, 360), Image.Resampling.NEAREST)